In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import itertools
import random


# Loading all the datasets for all days.

file_paths = [
    "../data/raw/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "../data/raw/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
    "../data/raw/Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "../data/raw/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "../data/raw/Monday-WorkingHours.pcap_ISCX.csv",
    "../data/raw/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "../data/raw/Tuesday-WorkingHours.pcap_ISCX.csv",
    "../data/raw/Wednesday-workingHours.pcap_ISCX.csv"
]

dfs = []
for path in file_paths:
    df_temp = pd.read_csv(path)
    df_temp.columns = df_temp.columns.str.strip()
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)

print("Dataset shape:", df.shape)
print(df["Label"].value_counts())

# Preprocessing for model training
# Adding Binary labels for benign & attack
df["Label_binary"] = df["Label"].apply(lambda x: 0 if x == "BENIGN" else 1)

X = df.drop(columns=["Label", "Label_binary"])
y = df["Label_binary"].values

# Spliting in training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Clipping extreme values
X_train = X_train.clip(-1e12, 1e12)
X_test = X_test.clip(-1e12, 1e12)

# Handling missing values
imputer = SimpleImputer(strategy="median")
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Scaling (for NN only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

feature_names = X.columns


# Random Forest

rf_base = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

rf_base.fit(X_train_imp, y_train)

importances = pd.Series(rf_base.feature_importances_, index=feature_names)
importances = importances.sort_values(ascending=False)


# Feature selection

TOP_K = 20
top_features = importances.head(TOP_K).index

X_train_fs = pd.DataFrame(X_train_imp, columns=feature_names)[top_features]
X_test_fs = pd.DataFrame(X_test_imp, columns=feature_names)[top_features]

print("Reduced shape:", X_train_fs.shape)


# Random Forest Hyperparameter Tuning
param_grid_rf = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight="balanced"),
    param_grid_rf,
    cv=3,
    scoring="f1",
    n_jobs=-1
)

grid_rf.fit(X_train_fs, y_train)

best_rf = grid_rf.best_estimator_
print("\nBest RF params:", grid_rf.best_params_)

# RF predictions
rf_pred = best_rf.predict(X_test_fs)


X_train_nn = X_train_fs.values.astype(np.float32)
X_test_nn = X_test_fs.values.astype(np.float32)

y_train_nn = y_train.astype(np.float32).reshape(-1, 1)
y_test_nn = y_test.astype(np.float32).reshape(-1, 1)

train_dataset = TensorDataset(
    torch.tensor(X_train_nn),
    torch.tensor(y_train_nn)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_nn),
    torch.tensor(y_test_nn)
)


# Neural Network

class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),

            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        return self.net(x)


# Hyperparameter tuning through Random search for NN

param_space = {
    "lr": [0.001, 0.0005],
    "batch_size": [128, 256],
    "hidden": [64, 128]
}

configs = list(itertools.product(
    param_space["lr"],
    param_space["batch_size"],
    param_space["hidden"]
))

configs = random.sample(configs, 3)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_f1 = 0
best_nn_model = None

criterion = nn.BCEWithLogitsLoss()

for lr, batch_size, hidden in configs:

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = NeuralNet(X_train_fs.shape[1], hidden).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # Train (few epochs for tuning)
    for epoch in range(5):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    # Evaluation
    model.eval()
    preds = []

    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            probs = torch.sigmoid(model(X_batch))
            preds.extend((probs >= 0.5).int().cpu().numpy())

    preds = np.array(preds).ravel()

    f1 = f1_score(y_test, preds)

    print(f"Config lr={lr}, batch={batch_size}, hidden={hidden} → F1={f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        best_nn_model = model
        best_nn_preds = preds


nn_pred = best_nn_preds

results = pd.DataFrame({
    "Model": ["Random Forest", "Neural Network"],
    "Accuracy": [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, nn_pred)
    ],
    "Precision": [
        precision_score(y_test, rf_pred),
        precision_score(y_test, nn_pred)
    ],
    "Recall": [
        recall_score(y_test, rf_pred),
        recall_score(y_test, nn_pred)
    ],
    "F1-score": [
        f1_score(y_test, rf_pred),
        f1_score(y_test, nn_pred)
    ]
})

print("\n===== FINAL RESULTS =====")
print(results)